In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.datasets import fetch_openml

Загрузка датасета Titanic из scikit-learn

X - признаки объекта, входные данные: таблица с характеристиками пассажиров (возраст, пол, класс билета, стоимость билета)

y - целевая переменная, то, что модель должна предсказать: столбец survived, выжил пассажир или нет

In [ ]:
X, y = fetch_openml("titanic", version=1, as_frame=True, return_X_y=True)

Объединяем признаки и целевой столбец

In [ ]:
df = X.copy()
df["survived"] = y

Размер датасета

In [ ]:
print(df.shape)

(1309, 14)


Первые 5 строк

In [ ]:
print(df.head())

   pclass                                             name     sex      age  \
0       1                    Allen, Miss. Elisabeth Walton  female  29.0000   
1       1                   Allison, Master. Hudson Trevor    male   0.9167   
2       1                     Allison, Miss. Helen Loraine  female   2.0000   
3       1             Allison, Mr. Hudson Joshua Creighton    male  30.0000   
4       1  Allison, Mrs. Hudson J C (Bessie Waldo Daniels)  female  25.0000   

   sibsp  parch  ticket      fare    cabin embarked boat   body  \
0      0      0   24160  211.3375       B5        S    2    NaN   
1      1      2  113781  151.5500  C22 C26        S   11    NaN   
2      1      2  113781  151.5500  C22 C26        S  NaN    NaN   
3      1      2  113781  151.5500  C22 C26        S  NaN  135.0   
4      1      2  113781  151.5500  C22 C26        S  NaN    NaN   

                         home.dest survived  
0                     St Louis, MO        1  
1  Montreal, PQ / Chesterville

Типы данных

In [ ]:
print(df.dtypes)

pclass          int64
name           object
sex          category
age           float64
sibsp           int64
parch           int64
ticket         object
fare          float64
cabin          object
embarked     category
boat           object
body          float64
home.dest      object
survived     category
dtype: object


Количество пропусков по столбцам

In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0])

age           263
fare            1
cabin        1014
embarked        2
boat          823
body         1188
home.dest     564
dtype: int64


Выберем признаки

Числовые признаки

In [ ]:
numeric_features = ["age", "fare"]

Категориальные признаки

In [ ]:
categorical_features = ["embarked", "sex", "pclass"]

Созадем таблицу с нужными столбцами

In [ ]:
X_lab = X[numeric_features + categorical_features].copy()

Используемые признаки

In [ ]:
print(X_lab.head())

       age      fare embarked     sex  pclass
0  29.0000  211.3375        S  female       1
1   0.9167  151.5500        S    male       1
2   2.0000  151.5500        S  female       1
3  30.0000  151.5500        S    male       1
4  25.0000  151.5500        S  female       1


Пропуски до обработки: столбцы и количество

In [ ]:
print(X_lab.isnull().sum())

age         263
fare          1
embarked      2
sex           0
pclass        0
dtype: int64


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

Pipeline для числовых признаков:
- заполнение пропусков медианой
- масштабирование StandardScaler

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

Pipeline для категориальных признаков:
- заполнение пропусков самым частым значением
- one-hot кодирование (превращаем категориальный признак в набор числовых столбцов 0 и 1)

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

Общий преобразователь

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

Обучаем преобразователь и преобразуем данные

In [ ]:
X_processed = preprocessor.fit_transform(X_lab)

Размер данных после обработки (строки, столбцы)

In [ ]:
print(X_processed.shape)

(1309, 10)


Получим имена новых признаков

In [ ]:
cat_feature_names = preprocessor.named_transformers_["cat"] \
    .named_steps["encoder"] \
    .get_feature_names_out(categorical_features)

In [ ]:
all_feature_names = numeric_features + list(cat_feature_names)

Переведем результат в dataFrame для просмотра

In [ ]:
if hasattr(X_processed, "toarray"):
    X_processed_dense = X_processed.toarray()
else:
    X_processed_dense = X_processed

In [ ]:
processed_df = pd.DataFrame(X_processed_dense, columns=all_feature_names)

Первые 5 строк после обработки

In [ ]:
print(processed_df.head())

        age      fare  embarked_C  embarked_Q  embarked_S  sex_female  \
0 -0.039005  3.442584         0.0         0.0         1.0         1.0   
1 -2.215952  2.286639         0.0         0.0         1.0         0.0   
2 -2.131977  2.286639         0.0         0.0         1.0         1.0   
3  0.038512  2.286639         0.0         0.0         1.0         0.0   
4 -0.349075  2.286639         0.0         0.0         1.0         1.0   

   sex_male  pclass_1  pclass_2  pclass_3  
0       0.0       1.0       0.0       0.0  
1       1.0       1.0       0.0       0.0  
2       0.0       1.0       0.0       0.0  
3       1.0       1.0       0.0       0.0  
4       0.0       1.0       0.0       0.0  


Есть ли пропуски после обработки?

In [ ]:
print(processed_df.isnull().sum().sum())

0


Названия признаков после кодирования

In [ ]:
print(processed_df.columns.tolist())

['age', 'fare', 'embarked_C', 'embarked_Q', 'embarked_S', 'sex_female', 'sex_male', 'pclass_1', 'pclass_2', 'pclass_3']
